# 8 · Rating models, and the "rising above yourself" test

Turns ball values into player ratings, with several methods (a ridge plus-minus model, empirical Bayes variants, a
combined profile). It also runs the key test for the brief's "compared with their usual level": is performing better
under pressure than your own normal a repeatable skill? (In this data, it is not.)

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes</summary>

P6 - Player rating models (R-1 ridge APM, R-2 empirical Bayes x3, R-3 composite) fitted on all data.  
Also: the clutch-spread test (is there real between-player variance in the pressure-vs-usual gap?),  
model agreement (Kendall tau), and a grouped-CV check of the ridge penalties.
</details>

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from rrlib.common import CFG, load, save_table, save_json, load_json, get_logger, Timer
from rrlib import ratings as RL

log = get_logger("p06_ratings")
v = load("values")
players = load("players").set_index("player_id")
comp = load_json("p05_components")

summary = {}
all_tabs = {}
for side in ("bat", "bowl"):
    tabs = {}
    with Timer(log, f"{side} ratings"):
        for met in ("v1", "v2", "v3", "v4"):
            d = RL.side_frame(v, side, met)
            e, k, tau2 = RL.eb_shrink(d)
            dl, tau_d, mu_d = RL.delta_shrink(d)
            ep, kp, tau2p = RL.eb_shrink(d[d.pres])
            summary[f"{side}_{met}"] = {"k_all": k, "tau_all": np.sqrt(tau2), "k_pressure": kp, "tau_pressure": np.sqrt(tau2p),
                                        "tau_clutch": np.sqrt(tau_d), "league_gap": mu_d,
                                        "sigma_ball": float(d.val.std()),
                                        "clutch_share_of_pressure_var": tau_d / max(tau2p, 1e-12)}
        for model in ("R2-p_only", "R2-joint", "R2-weighted"):
            tabs[model] = RL.r2_ratings(v, side, "v1", model.split("-")[1])
        tabs["R1"] = RL.r1_ratings(v, side, "v1")
        tabs["R3"] = RL.r3_ratings(v, side, comp)
    # agreement among players with >= 60 pressure balls
    base = tabs["R2-joint"]
    pool = base.index[base.n_p >= 60]
    names = list(tabs)
    M = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b_ in names:
            x, y = tabs[a].rating.reindex(pool), tabs[b_].rating.reindex(pool)
            okk = x.notna() & y.notna()
            M.loc[a, b_] = kendalltau(x[okk], y[okk])[0]
    save_table(M.reset_index().rename(columns={"index": "model"}), f"rating_agreement_{side}")
    log.info(f"{side} model agreement (Kendall tau, players with >=60 pressure balls, n={len(pool)}):\n{M.round(2).to_string()}")
    # combined table
    T = pd.DataFrame(index=base.index)
    T["name"] = players.name.reindex(T.index)
    T["n_pressure"] = base.n_p; T["n_other"] = base.n_u
    for nm, t in tabs.items():
        T[nm] = t.rating.reindex(T.index)
    T["R1_usual"] = tabs["R1"].usual.reindex(T.index); T["R1_clutch"] = tabs["R1"].clutch.reindex(T.index)
    T["gap_raw"] = base.gap_raw; T["gap_shrunk"] = base.gap_shrunk; T["gap_reliability"] = base.gap_rel
    T = T.sort_values("R2-joint", ascending=False)
    save_table(T.reset_index().rename(columns={"index": "player_id", "pid": "player_id"}), f"ratings_{side}_all_models")
    all_tabs[side] = T
    log.info(f"{side}: top 12 by R2-joint\n" + T[T.n_pressure >= 30].head(12)[["name", "n_pressure", "R2-joint", "R1", "R3", "R2-p_only", "gap_shrunk"]].round(3).to_string())

S = pd.DataFrame(summary).T
save_table(S.reset_index().rename(columns={"index": "side_metric"}), "clutch_spread")
log.info("Variance decomposition (per ball, runs or WP units):\n" + S.round(4).to_string())

13:40:22 [bat ratings] 2.1s


13:40:22 bat model agreement (Kendall tau, players with >=60 pressure balls, n=174):
             R2-p_only  R2-joint  R2-weighted    R1    R3
R2-p_only         1.00      0.38         0.68  0.39  0.29
R2-joint          0.38      1.00         0.55  0.92  0.13
R2-weighted       0.68      0.55         1.00  0.56  0.26
R1                0.39      0.92         0.56  1.00  0.12
R3                0.29      0.13         0.26  0.12  1.00


13:40:22 bat: top 12 by R2-joint
                       name  n_pressure  R2-joint     R1     R3  R2-p_only  gap_shrunk
pid                                                                                   
50284  Vaibhav Sooryavanshi       198.0     0.282  0.261 -0.001      0.063        -0.0
6396              Tim David       287.0     0.230  0.212 -0.001      0.057        -0.0
6281        Abhishek Sharma       497.0     0.209  0.193 -0.006      0.037        -0.0
1318          Andre Russell       102.0     0.174  0.153 -0.006      0.012        -0.0
4260       Heinrich Klaasen       177.0     0.171  0.148  0.007      0.062         0.0
5039          Rajat Patidar       164.0     0.163  0.148 -0.003      0.021        -0.0
6726       Romario Shepherd       128.0     0.150  0.130  0.010      0.013        -0.0
1699       Suryakumar Yadav       558.0     0.145  0.115  0.011      0.071         0.0
143          Dinesh Karthik       132.0     0.127  0.093  0.004      0.017        -0.0
3125      

13:40:24 [bowl ratings] 2.0s


13:40:24 bowl model agreement (Kendall tau, players with >=60 pressure balls, n=191):
             R2-p_only  R2-joint  R2-weighted    R1    R3
R2-p_only         1.00      0.35         0.73  0.35  0.47
R2-joint          0.35      1.00         0.48  0.93  0.27
R2-weighted       0.73      0.48         1.00  0.47  0.44
R1                0.35      0.93         0.47  1.00  0.27
R3                0.47      0.27         0.44  0.27  1.00


13:40:24 bowl: top 12 by R2-joint
                     name  n_pressure  R2-joint     R1     R3  R2-p_only  gap_shrunk
pid                                                                                 
2933       Jasprit Bumrah       396.0     0.151  0.194  0.039      0.044        -0.0
6256   Varun Chakravarthy       442.0     0.100  0.146  0.012     -0.012        -0.0
7920         Nathan Ellis       228.0     0.098  0.142  0.019      0.006        -0.0
4641          Lungi Ngidi       136.0     0.098  0.141  0.014     -0.004        -0.0
34485      Sufiyan Muqeem       132.0     0.090  0.124  0.028      0.039         0.0
781          Sunil Narine       338.0     0.088  0.137  0.034      0.032         0.0
4856       Benjamin Sears       110.0     0.082  0.113  0.002      0.068         0.0
3644        Kuldeep Yadav       322.0     0.080  0.124  0.015     -0.014        -0.0
830        Josh Hazlewood       366.0     0.064  0.104  0.020      0.035         0.0
4722          Rashid Khan      

13:40:24 Variance decomposition (per ball, runs or WP units):
             k_all  tau_all  k_pressure  tau_pressure  tau_clutch  league_gap  sigma_ball  clutch_share_of_pressure_var
bat_v1    627.6459   0.0970   1197.5154        0.0625       0.000     -0.0040      2.4304                        0.0000
bat_v2    981.3184   0.0010   1112.3881        0.0017       0.000      0.0002      0.0299                        0.0000
bat_v3    427.3547   0.0399    380.0623        0.0436       0.000     -0.0059      0.8242                        0.0000
bat_v4    566.7778   0.0824    627.4660        0.0708       0.000     -0.0087      1.9619                        0.0000
bowl_v1   986.1439   0.0767    924.3832        0.0705       0.000     -0.0073      2.4077                        0.0000
bowl_v2  1208.8821   0.0009    475.3480        0.0026       0.003     -0.0002      0.0297                        1.3344
bowl_v3   444.0690   0.0261    424.1087        0.0274       0.000      0.0042      0.5494         

#### Ridge penalty check: grouped 5-fold CV of held-out ball value, multipliers of the Bayesian penalty

In [3]:
folds = load("folds").set_index("match_id").fold
v["fold"] = v.match_id.map(folds)
rows = []
with Timer(log, "R1 penalty CV"):
    for km in (0.25, 0.5, 1.0, 2.0, 4.0):
        se, se0 = 0.0, 0.0
        for k in range(5):
            tr, te = v[v.fold != k], v[v.fold == k]
            fit = RL.r1_fit(tr, "v1", km)
            yb = -te.v1_bowl.values
            yb = yb - (-tr.v1_bowl.values).mean()
            pb = (fit["bat"].usual.reindex(te.striker_id).fillna(0).values * te.faced.values
                  + fit["bat"].clutch.reindex(te.striker_id).fillna(0).values * te.faced.values * te.is_pressure.values
                  - fit["bowl"].usual.reindex(te.bowler_id).fillna(0).values
                  - fit["bowl"].clutch.reindex(te.bowler_id).fillna(0).values * te.is_pressure.values)
            se += ((yb - pb) ** 2).sum(); se0 += (yb ** 2).sum()
        rows.append({"penalty_multiplier": km, "heldout_mse_reduction_pct": 100 * (1 - se / se0)})
        log.info(f"R1 penalty x{km}: held-out MSE reduction {100 * (1 - se / se0):.4f}%")
save_table(pd.DataFrame(rows), "r1_penalty_cv")
save_json(summary, "p06_summary")
log.info("P6 done")

13:40:27 R1 penalty x0.25: held-out MSE reduction 0.0025%


13:40:29 R1 penalty x0.5: held-out MSE reduction 0.0457%


13:40:31 R1 penalty x1.0: held-out MSE reduction 0.0632%


13:40:34 R1 penalty x2.0: held-out MSE reduction 0.0597%


13:40:36 R1 penalty x4.0: held-out MSE reduction 0.0450%


13:40:36 [R1 penalty CV] 11.9s


13:40:36 P6 done
